# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Dataset DOI:** [10.71728/senscience.y7m0-f273](https://sen.science/doi/10.71728/senscience.y7m0-f273)
- **Schema URL:** https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
- **Coverage:** 475 pastoralist households across Samburu, Isiolo, and Marsabit counties, Northern Kenya


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Date Published: {getattr(metadata, 'datePublished', 'Not provided')}")
print(f"License: {getattr(metadata, 'license', 'Not provided')}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")
print(f"Spatial Coverage: {getattr(metadata, 'spatialCoverage', 'Not provided')}")
print(f"Temporal Coverage: {getattr(metadata, 'temporalCoverage', 'Not provided')}")

## 2. Data Overview
List available record sets, fields, and their `@id` values for further exploration.


In [ ]:
# List all record sets in the dataset
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"- Record Set name: {getattr(rs, 'name', 'N/A')} | @id: {rs.id}")

# Show the fields (columns) of each record set
for rs in record_sets:
    print(f"\nRecord Set: {getattr(rs, 'name', 'N/A')} | @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            dtype = getattr(field, 'dataType', None)
            print(f"  - Field: {getattr(field, 'name', 'N/A'):<30} @id: {field.id}  dataType: {dtype}")
    else:
        print("  [No fields available]")

## 3. Data Extraction
Load data from each record set as a Pandas DataFrame for analysis using their `@id` values. This allows for referencing and manipulation of fields and columns by their unique `@id`s. 

We will extract the tabular content of each record set.

In [ ]:
# Collect all record set IDs for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f'Extracting records for record set: {record_set_id}')
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            print(f"  - Number of rows: {len(df)}  | Columns: {df.columns.tolist()}")
            dataframes[record_set_id] = df
        else:
            print("  [No records found]")
    except Exception as e:
        print(f"  [Failed to extract: {e}]")

# Preview the first available DataFrame
if dataframes:
    # Use the first non-empty dataframe
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nSample of data from record set {first_rs_id}:")
    print(dataframes[first_rs_id].head())
else:
    print("No record set data available for analysis.")


## 4. Exploratory Data Analysis (EDA)

Let's demonstrate basic EDA: filtering records, normalizing numeric fields, and grouping data. **All field references use their Croissant `@id`.**

- Select a numeric field (e.g., a regression coefficient column) from the overview above for filtering and normalization.
- Choose a grouping variable (e.g., a categorical field such as intervention type or location) if available.


In [ ]:
# Find the first DataFrame with at least one numeric column for EDA
import numpy as np
eda_rs_id = None
numeric_field_id = None
group_field_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        # Try to infer numeric columns (float or int)
        for col in df.columns:
            try:
                col_numeric = pd.to_numeric(df[col], errors='coerce')
                if np.isfinite(col_numeric).sum() > 0:
                    numeric_field_id = col
                    eda_rs_id = rs_id
                    break
            except:
                continue
    if numeric_field_id is not None:
        break

# Suggest a group field (first string/categorical column)
if eda_rs_id is not None:
    df = dataframes[eda_rs_id]
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == 'object' and df[col].nunique() < 10 and df[col].nunique() > 1:
            group_field_id = col
            break
    
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = numeric_series.dropna().mean()  # use mean as a simple threshold
    filtered_df = df[numeric_series > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df[[numeric_field_id]].head())

    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series - numeric_series.mean()) / numeric_series.std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields (columns) in the selected record set. We'll plot a histogram of the numeric field and, if a group is found, a bar plot of the group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if eda_rs_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(dataframes[eda_rs_id][numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {eda_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in dataframes[eda_rs_id].columns:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=dataframes[eda_rs_id], ci=None)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No suitable data to visualize.")

## 6. Conclusion

- Successfully explored the FAIR² dataset using `mlcroissant`.
- Loaded metadata, identified record sets and their fields by `@id`, and loaded the main tabular data into pandas DataFrames.
- Demonstrated basic filtering, normalization, grouping, and data visualization.

### Next Steps
- Explore field-level documentation in the Croissant schema for semantic data understanding.
- Perform more advanced analyses (e.g., regression, factor analysis) using the rich dataset fields referenced by their `@id`s.

**Reference:** Kamadi, V., Chimoita, E.L., Wahome, R.G., Odhong, C. (2026). Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya.
